In [76]:
import pandas as pd
import numpy as np

In [77]:
data = pd.read_csv("global-data-on-sustainable-energy.csv")
data.head()

,Entity,Year,Access to electricity (% of population),Access to clean fuels for cooking,Renewable-electricity-generating-capacity-per-capita,Financial flows to developing countries (US $),Renewable energy share in the total final energy consumption (%),Electricity from fossil fuels (TWh),Electricity from nuclear (TWh),Electricity from renewables (TWh),...,Primary energy consumption per capita (kWh/person),Energy intensity level of primary energy (MJ/$2017 PPP GDP),Value_co2_emissions_kt_by_country,Renewables (% equivalent primary energy),gdp_growth,gdp_per_capita,Density\n(P/Km2),Land Area(Km2),Latitude,Longitude
0,Afghanistan,2000,1.613591,6.2,9.22,20000.0,44.99,0.16,0.0,0.31,...,302.59482,1.64,760.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
1,Afghanistan,2001,4.074574,7.2,8.86,130000.0,45.60,0.09,0.0,0.50,...,236.89185,1.74,730.000000,NaN,NaN,NaN,60,652230.0,33.93911,67.709953
2,Afghanistan,2002,9.409158,8.2,8.47,3950000.0,37.83,0.13,0.0,0.56,...,210.86215,1.40,1029.999971,NaN,NaN,179.426579,60,652230.0,33.93911,67.709953
3,Afghanistan,2003,14.738506,9.5,8.09,25970000.0,36.66,0.31,0.0,0.63,...,229.96822,1.40,1220.000029,NaN,8.832278,190.683814,60,652230.0,33.93911,67.709953
4,Afghanistan,2004,20.064968,10.9,7.75,NaN,44.24,0.33,0.0,0.56,...,204.23125,1.20,1029.999971,NaN,1.414118,211.382074,60,652230.0,33.93911,67.709953


In [78]:
data.isnull().sum()

Entity                                                                 0
Year                                                                   0
Access to electricity (% of population)                               10
Access to clean fuels for cooking                                    169
Renewable-electricity-generating-capacity-per-capita                 931
Financial flows to developing countries (US $)                      2089
Renewable energy share in the total final energy consumption (%)     194
Electricity from fossil fuels (TWh)                                   21
Electricity from nuclear (TWh)                                       126
Electricity from renewables (TWh)                                     21
Low-carbon electricity (% electricity)                                42
Primary energy consumption per capita (kWh/person)                     0
Energy intensity level of primary energy (MJ/$2017 PPP GDP)          207
Value_co2_emissions_kt_by_country                  

In [79]:
a = np.zeros(21, dtype="int") 
y = np.zeros(21, dtype="int")


for i in range(21):
    a[i] = len(data[data["Year"] == 2000+i])
    y[i] = 2000+i

pd.DataFrame({"Year":y, "Amount":a})


,Year,Amount
0,2000,173
1,2001,172
2,2002,172
3,2003,172
4,2004,172
5,2005,172
6,2006,172
7,2007,174
8,2008,174
9,2009,174


In [80]:
data["Entity"].value_counts()

Entity
Afghanistan            21
Albania                21
Algeria                21
Angola                 21
Antigua and Barbuda    21
                       ..
Uzbekistan             21
Serbia                 14
Montenegro             14
South Sudan             8
French Guiana           1
Name: count, Length: 176, dtype: int64

# Preprocessing

In [81]:
# Instantly drop columns that has missing GDP Per Capita
ignored = data[pd.isna(data["gdp_per_capita"])]
data = data[pd.notna(data["gdp_per_capita"])]

ignored["Entity"].value_counts()

Entity
Congo                               21
Bahamas                             21
Czechia                             21
Saint Kitts and Nevis               21
Gambia                              21
Egypt                               21
Yemen                               21
Slovakia                            21
Saint Vincent and the Grenadines    21
Kyrgyzstan                          21
Saint Lucia                         21
Somalia                             13
Nauru                               10
Eritrea                              9
Cayman Islands                       6
South Sudan                          5
Afghanistan                          2
Aruba                                2
New Caledonia                        1
French Guiana                        1
Sao Tome and Principe                1
Turkmenistan                         1
Name: count, dtype: int64

### Cluster Weight Equalizing

In [82]:
entity_counts = data["Entity"].value_counts()

eligible_entities = entity_counts[entity_counts == 21].index
used_df = data[data["Entity"].isin(eligible_entities)]

ineligible_entities = entity_counts[entity_counts != 21].index
unused_df = data[data["Entity"].isin(ineligible_entities)]

In [83]:
unused_df["Entity"].value_counts().sort_index()

Entity
Afghanistan              19
Aruba                    19
Cayman Islands           15
Eritrea                  12
Montenegro               14
Nauru                    11
New Caledonia            20
Sao Tome and Principe    20
Serbia                   14
Somalia                   8
South Sudan               3
Turkmenistan             20
Name: count, dtype: int64

In [84]:
print("Used Entity Count: ", len(used_df["Entity"].unique()))
used_df["Entity"].value_counts().sort_index()

Used Entity Count:  152


Entity
Albania                21
Algeria                21
Angola                 21
Antigua and Barbuda    21
Argentina              21
                       ..
Uruguay                21
Uzbekistan             21
Vanuatu                21
Zambia                 21
Zimbabwe               21
Name: count, Length: 152, dtype: int64

## Missing Value Imputation

In [85]:
used_df.isnull().sum()

Entity                                                                 0
Year                                                                   0
Access to electricity (% of population)                                9
Access to clean fuels for cooking                                    105
Renewable-electricity-generating-capacity-per-capita                 861
Financial flows to developing countries (US $)                      1773
Renewable energy share in the total final energy consumption (%)     171
Electricity from fossil fuels (TWh)                                   21
Electricity from nuclear (TWh)                                       126
Electricity from renewables (TWh)                                     21
Low-carbon electricity (% electricity)                                42
Primary energy consumption per capita (kWh/person)                     0
Energy intensity level of primary energy (MJ/$2017 PPP GDP)          151
Value_co2_emissions_kt_by_country                  

In [86]:
used_df.drop(["Financial flows to developing countries (US $)",
               "Renewables (% equivalent primary energy)"], axis = 1, inplace = True) # Too many missing values
used_df.drop(['gdp_growth'], axis=1, inplace=True)

/tmp/ipykernel_950/2268370681.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  used_df.drop(["Financial flows to developing countries (US $)",
/tmp/ipykernel_950/2268370681.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  used_df.drop(['gdp_growth'], axis=1, inplace=True)


In [87]:
used_df.isnull().sum()

Entity                                                                0
Year                                                                  0
Access to electricity (% of population)                               9
Access to clean fuels for cooking                                   105
Renewable-electricity-generating-capacity-per-capita                861
Renewable energy share in the total final energy consumption (%)    171
Electricity from fossil fuels (TWh)                                  21
Electricity from nuclear (TWh)                                      126
Electricity from renewables (TWh)                                    21
Low-carbon electricity (% electricity)                               42
Primary energy consumption per capita (kWh/person)                    0
Energy intensity level of primary energy (MJ/$2017 PPP GDP)         151
Value_co2_emissions_kt_by_country                                   195
gdp_per_capita                                                  

In [88]:
def fillBlankByEntityyMean(colName):
    meanByEntity = used_df.groupby('Entity')[colName].transform('mean')
    used_df[colName].fillna(meanByEntity, inplace = True)

In [89]:
fillBlankByEntityyMean('Electricity from fossil fuels (TWh)')
fillBlankByEntityyMean('Electricity from nuclear (TWh)')
fillBlankByEntityyMean('Electricity from renewables (TWh)')
fillBlankByEntityyMean('Low-carbon electricity (% electricity)')
fillBlankByEntityyMean('Value_co2_emissions_kt_by_country')
fillBlankByEntityyMean('Access to clean fuels for cooking')
fillBlankByEntityyMean('Renewable energy share in the total final energy consumption (%)')
fillBlankByEntityyMean('Energy intensity level of primary energy (MJ/$2017 PPP GDP)')

used_df['Access to electricity (% of population)'].fillna(0, inplace = True)
used_df['Renewable-electricity-generating-capacity-per-capita'].fillna(0, inplace = True)

/tmp/ipykernel_950/2194767598.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  used_df[colName].fillna(meanByEntity, inplace = True)
/tmp/ipykernel_950/2194767598.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  used_df[colName].fillna(meanByEntity, inplace = True)
/tmp/ipykernel_950/2194767598.py:3: FutureWarning: A value is trying to b

In [90]:
used_df.isnull().sum()

Entity                                                                0
Year                                                                  0
Access to electricity (% of population)                               0
Access to clean fuels for cooking                                   105
Renewable-electricity-generating-capacity-per-capita                  0
Renewable energy share in the total final energy consumption (%)     21
Electricity from fossil fuels (TWh)                                  21
Electricity from nuclear (TWh)                                      126
Electricity from renewables (TWh)                                    21
Low-carbon electricity (% electricity)                               42
Primary energy consumption per capita (kWh/person)                    0
Energy intensity level of primary energy (MJ/$2017 PPP GDP)           0
Value_co2_emissions_kt_by_country                                    42
gdp_per_capita                                                  

All of those are multiply of 21 which means for that entity, every single values for that column doesn't exists. Because there is no better way to fill, dropping the rows is a valid option because of a wide variance.<br><br>

In [91]:
print(used_df.shape)
used_df = used_df.dropna()
print(used_df.shape)
used_df.isnull().sum()

(3192, 18)
(2919, 18)


Entity                                                              0
Year                                                                0
Access to electricity (% of population)                             0
Access to clean fuels for cooking                                   0
Renewable-electricity-generating-capacity-per-capita                0
Renewable energy share in the total final energy consumption (%)    0
Electricity from fossil fuels (TWh)                                 0
Electricity from nuclear (TWh)                                      0
Electricity from renewables (TWh)                                   0
Low-carbon electricity (% electricity)                              0
Primary energy consumption per capita (kWh/person)                  0
Energy intensity level of primary energy (MJ/$2017 PPP GDP)         0
Value_co2_emissions_kt_by_country                                   0
gdp_per_capita                                                      0
Density\n(P/Km2)    

### Dtype correction

In [92]:
used_df.dtypes

Entity                                                               object
Year                                                                  int64
Access to electricity (% of population)                             float64
Access to clean fuels for cooking                                   float64
Renewable-electricity-generating-capacity-per-capita                float64
Renewable energy share in the total final energy consumption (%)    float64
Electricity from fossil fuels (TWh)                                 float64
Electricity from nuclear (TWh)                                      float64
Electricity from renewables (TWh)                                   float64
Low-carbon electricity (% electricity)                              float64
Primary energy consumption per capita (kWh/person)                  float64
Energy intensity level of primary energy (MJ/$2017 PPP GDP)         float64
Value_co2_emissions_kt_by_country                                   float64
gdp_per_capi

In [93]:
used_df['Density\\n(P/Km2)'] = used_df['Density\\n(P/Km2)'].str.replace(",", ".", regex=False).astype(float)

In [94]:
used_df.dtypes

Entity                                                               object
Year                                                                  int64
Access to electricity (% of population)                             float64
Access to clean fuels for cooking                                   float64
Renewable-electricity-generating-capacity-per-capita                float64
Renewable energy share in the total final energy consumption (%)    float64
Electricity from fossil fuels (TWh)                                 float64
Electricity from nuclear (TWh)                                      float64
Electricity from renewables (TWh)                                   float64
Low-carbon electricity (% electricity)                              float64
Primary energy consumption per capita (kWh/person)                  float64
Energy intensity level of primary energy (MJ/$2017 PPP GDP)         float64
Value_co2_emissions_kt_by_country                                   float64
gdp_per_capi

In [95]:
print("Final Entity Count: ", len(used_df["Entity"].unique()))
print("Data Shape: ", used_df.shape)
used_df["Entity"].value_counts().sort_index()

Final Entity Count:  139
Data Shape:  (2919, 18)


Entity
Algeria                21
Angola                 21
Antigua and Barbuda    21
Argentina              21
Armenia                21
                       ..
Uruguay                21
Uzbekistan             21
Vanuatu                21
Zambia                 21
Zimbabwe               21
Name: count, Length: 139, dtype: int64

## Saving into a csv file

In [96]:
used_df.to_csv('cleaned_dataset.csv', index=False)